# Seq2seq
<img src="./seq2seq.png" style="width: 80%;">  

- Sequence-to-sequence (Seq2seq) 모델은 기계 번역, 문서 요약, 그리고 이미지 캡셔닝 등의 문제에서 아주 큰 성공을 거둔 딥러닝 모델
- 글자, 단어, 이미지의 feature 등의 아이템 시퀀스를 입력으로 받아 또 다른 아이템의 시퀀스 출력

## model architecture
- encoder 1개 + decoder 1개 
    - **Encoder**: 입력 문장을 한 글자씩 처리하여 정보 추출 -> 하나의 벡터로 만듦(context vector)
    - **Decoder**: 인코더가 준 context vector를 바탕으로, 한 단어씩 목표 언어(예: 영어)를 생성
- Encoder, Decoder에서는 둘 다 RNN을 사용하는 경우가 많음
    - 왜 seq2seq가 필요한가? (encoder, decoder를 별도로 사용해야하나?) 
        - 입력과 출력의 길이가 다름. (나는 학교에 간다(3글자) -> I go to school(4글자))
        - 기존 RNN은 입력 1개 - 출력 1개 (Many-to-Many)
        - 입출력의 길이가 다르기 때문에 seq2seq는 인코더에서 문장을 다 읽을 때까지 기다렸다가(압축), 디코더에서 새로운 길이를 생성해내기 때문에 이 문제 해결

- **context** : float로 이루어진 하나의 벡터 
    - 벡터 크기는 모델 설정 시 원하는 값으로 설정 가능(보통 RNN의 hidden unit 개수)

- **word embedding** : 단어들을 벡터 공간에 투영하여 단어 간 관계와 의미 정보 파악 
    - One-hot encoding: 단어 사이의 관계를 모름!! (king-queen은 남남)
    - word embedding: 단어를 N차원 연속적인 숫자 공간에 뿌려 비슷한 의미의 단어끼리 모음 (king - man + woman = queen)
        - String(King) -> Integer(15): 사전에서 N번째 단어 -> embedding input(torch.tensor([15])) -> embedding output(크기 N의 vector)

- seq2seq에서 RNN의 입력 구조
    - RNN 셀 하나는 매 순간 현재 아이템의 벡터($x_t$) + 이전 상태의 기억($h_{t-1}$)을 받음
    - 이 두 벡터가 RNN 내부에서 합쳐져 지금까지의 문장 맥락을 담은 기억 생성 

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [3]:
# 샘플 데이터 (독일어 -> 영어)
data = [
    ["guten morgen", "good morning"],
    ["ich liebe dich", "i love you"],
    ["wie geht es dir", "how are you"]
]

In [7]:
# 단어장 만들기(Vocab) 

def build_vocab(sentences):
    vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3} 
    # <pad>:빈칸채우기/<sos>:시작/<eos>:끝/<unk>:모르는 단어
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

# 단어 -> 번호 ({"apple": 5})
de_vocab = build_vocab([pair[0] for pair in data])    
en_vocab = build_vocab([pair[1] for pair in data])

# 번호 -> 단어 ({5: "apple"})
en_idx_to_word = {i: w for w, i in en_vocab.items()} # input인 독일어만 인덱싱

# 문장 -> 숫자 변환 ()
def tokenize(text, vocab):
    # 문장의 시작 <sos> 를 리스트에 추가
    # 문장을 공백단위로 쪼개서 단어를 번호로 변경 
    # 문장의 끝을 알리는 번호 붙이기 
    # 텐서 형태로 변환
    ids = [vocab["<sos>"]] + [vocab.get(w, vocab["<unk>"]) for w in text.split()]
    return torch.LongTensor(ids)

train_pairs = [[tokenize(de, de_vocab), tokenize(en, en_vocab)] for de, en in data]

print("de_vocab", de_vocab)
print("en_vocab", en_vocab)

print(train_pairs)

de_vocab {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'guten': 4, 'morgen': 5, 'ich': 6, 'liebe': 7, 'dich': 8, 'wie': 9, 'geht': 10, 'es': 11, 'dir': 12}
en_vocab {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'good': 4, 'morning': 5, 'i': 6, 'love': 7, 'you': 8, 'how': 9, 'are': 10}
[[tensor([1, 4, 5]), tensor([1, 4, 5])], [tensor([1, 6, 7, 8]), tensor([1, 6, 7, 8])], [tensor([ 1,  9, 10, 11, 12]), tensor([ 1,  9, 10,  8])]]


In [ ]:
# Encoder & Decoder 

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hid_dim, n_layers)
        
    def forward(self, src):
        # src = [sentence length, 1] (문장, 배치)
        embedded = self.embedding(src)
        
        # RNN
        outputs, hidden = self.rnn(embedded)
        
        # 최종 hidden = context vector 
        return hidden 
    
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers):
        super().__init__()
        self.output_dim = output_dim # self를 붙이는 기준 -> 나중에 또 쓸 것인가?
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hid_dim, n_layers)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
    def forward(self, input, hidden):
        # input = [1] (단어 1개) -> [1, 1] (1글자, 1배치)
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        
        # 이전 단계의 hidden만 받아서 새 hidden 생성
        output, hidden = self.rnn(embedded, hidden)
        
        # 마지막 층의 hidden으로 단어 예측
        prediction = self.fc_out(output.unsqueeze(0))
        
        return prediction, hidden
        

In [10]:
# seq2seq model 
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder 
        self.decoder = decoder
        
    def forward(self, src, trg):
        # src: [독어 문장 길이, 배치크기(=1)]
        # trg: [영어 문장 길이, 배치크기(=1)]
        
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim 
        
        # 예측된 결과를 저장할 공간 (영어 문장 길이 만큼 준비)
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size)
        
        # 1. 인코더가 문장을 다 읽고 마지막 기억을 뱉음 
        hidden = self.encoder(src)
        
        # 2. 디코더의 첫번째 입력은 무조건 <sos>
        input = trg[0, :]
        
        # 3. 영어 문장 길이만큼 반복하며 단어 하나씩 생성
        for t in range(1, trg_len):
            # 디코더에 현재 단어와 이전 기억을 넣음
            output, hidden = self.decoder(input, hidden)
            
            # 예측 결과 저장
            outputs[t] = output
            
            top1 = output.argmax(1)
            
        return outputs
            

In [11]:
# 단어장 크기 설정
INPUT_DIM = len(de_vocab)
OUTPUT_DIM = len(en_vocab)

# 하이퍼파라미터 (단순하게 설정)
ENC_EMB_DIM = 32
DEC_EMB_DIM = 32
HID_DIM = 64
N_LAYERS = 2

# 모델 객체 생성
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS)
model = Seq2Seq(enc, dec)

# 최적화 도구 및 손실 함수
optimizer = optim.Adam(model.parameters())
# <pad> 토큰은 학습에서 무시하도록 ignore_index 설정
criterion = nn.CrossEntropyLoss(ignore_index=en_vocab["<pad>"])

In [13]:
model.train()
EPOCHS = 100

for epoch in range(EPOCHS):
    epoch_loss = 0
    
    for src, trg in train_pairs:
        # 모양 맞추기: [길이] -> [길이, 1] (배치 사이즈 1)
        src = src.unsqueeze(1)
        trg = trg.unsqueeze(1)
        
        optimizer.zero_grad()
        
        # 모델 실행
        output = model(src, trg)
        
        # output: [trg_len, 1, output_dim]
        # trg: [trg_len, 1]
        
        # Loss 계산을 위해 첫 <sos>는 제외하고 차원 펴주기
        output_dim = output.shape[-1]
        loss_output = output[1:].view(-1, output_dim)
        loss_trg = trg[1:].view(-1)
        
        loss = criterion(loss_output, loss_trg)
        loss.backward()
        
        # 기울기 폭주 방지 (Gradient Clipping)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 20 == 0:
        print(f"Epoch: {epoch+1:3} | Loss: {epoch_loss/len(train_pairs):.4f}")

Epoch:  20 | Loss: 0.1153
Epoch:  40 | Loss: 0.0257
Epoch:  60 | Loss: 0.0135
Epoch:  80 | Loss: 0.0087
Epoch: 100 | Loss: 0.0061


In [15]:
def translate_sentence(model, sentence, de_vocab, en_idx_to_word, max_len=10):
    model.eval()
    with torch.no_grad():
        # 1. 입력 문장 전처리
        src = tokenize(sentence, de_vocab).unsqueeze(1)
        
        # 2. 인코더 통과시켜 Context Vector(hidden) 획득
        hidden = model.encoder(src)
        
        # 3. 디코더 시작 준비 (<sos> 토큰)
        curr_input = torch.tensor([en_vocab["<sos>"]])
        result = []
        
        for _ in range(max_len):
            # 디코더 한 스텝 실행
            output, hidden = model.decoder(curr_input, hidden)
            
            # 1. argmax의 결과에서 첫 번째 값을 명시적으로 가져옵니다.
            # output이 [1, 1, vocab] 이라면 argmax(-1)은 [1, 1]이 됩니다.
            pred = output.argmax(-1).view(-1) # [1]로 펴주기
            
            # <eos> 나오면 종료
            if pred[0] == en_vocab["<eos>"]: 
                break
                
            # 결과 저장 및 다음 입력으로 사용
            result.append(en_idx_to_word[pred.item()])
            curr_input = pred
            
        return " ".join(result)

# 실행 예시
test_sentence = "ich liebe dich"
print(f"독일어: {test_sentence}")
print(f"번역결과: {translate_sentence(model, test_sentence, de_vocab, en_idx_to_word)}")


독일어: ich liebe dich
번역결과: i love you are you love you are you love
